# 01.5 · Pandas: missing values, duplicates, data types, dates

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/01-working-with-data/05-pandas-clean/lesson.ipynb)

Read the [README](README.md) first. Here we clean the six-row table from the README by hand, then the real ShopWave data.

## 1. The six rows

In [1]:
import pandas as pd
import numpy as np

six = pd.DataFrame({
    "invoice":  ["536365", "536365", "C536379", "536380", "536381", "536382"],
    "product":  ["WHITE LANTERN", "WHITE LANTERN", "MUG", None, "JAM JAR", "TEA TOWEL"],
    "qty":      [6, 6, -1, 2, 12, 3],
    "price":    [3.39, 3.39, 2.55, 0.00, "1.25", 1.65],
    "customer": [17850, 17850, 17850, None, 15311, 15311],
    "date":     ["2010-12-01", "2010-12-01", "2010-12-01", "2010-12-01", "01/12/2010", "2010-12-02"],
})
six

,invoice,product,qty,price,customer,date
0,536365,WHITE LANTERN,6,3.39,17850.0,2010-12-01
1,536365,WHITE LANTERN,6,3.39,17850.0,2010-12-01
2,C536379,MUG,-1,2.55,17850.0,2010-12-01
3,536380,NaN,2,0.0,NaN,2010-12-01
4,536381,JAM JAR,12,1.25,15311.0,01/12/2010
5,536382,TEA TOWEL,3,1.65,15311.0,2010-12-02


## 2. Question 1: what is empty?

`isna()` returns a True/False table the same shape as the data. Summing it counts the Trues per column; `.mean()` gives the fraction.

In [2]:
print(six.isna().sum())
print()
print("missing %:")
print(six.isna().mean().mul(100).round(1))

invoice     0
product     1
qty         0
price       0
customer    1
date        0
dtype: int64

missing %:
invoice      0.0
product     16.7
qty          0.0
price        0.0
customer    16.7
date         0.0
dtype: float64


Row 3 has no product and no customer. For a customer analysis it is unusable, so we drop rows with a missing customer. `subset=` says *which* columns decide.

In [3]:
six = six.dropna(subset=["customer"])
six

,invoice,product,qty,price,customer,date
0,536365,WHITE LANTERN,6,3.39,17850.0,2010-12-01
1,536365,WHITE LANTERN,6,3.39,17850.0,2010-12-01
2,C536379,MUG,-1,2.55,17850.0,2010-12-01
4,536381,JAM JAR,12,1.25,15311.0,01/12/2010
5,536382,TEA TOWEL,3,1.65,15311.0,2010-12-02


## 3. Question 2: what is counted twice?

`duplicated()` marks a row True if an identical row appeared earlier.

In [4]:
print(six.duplicated())
six = six.drop_duplicates()
six

0    False
1     True
2    False
4    False
5    False
dtype: bool


,invoice,product,qty,price,customer,date
0,536365,WHITE LANTERN,6,3.39,17850.0,2010-12-01
2,C536379,MUG,-1,2.55,17850.0,2010-12-01
4,536381,JAM JAR,12,1.25,15311.0,01/12/2010
5,536382,TEA TOWEL,3,1.65,15311.0,2010-12-02


## 4. Question 3: what is the wrong type?

Look at `dtypes`. `price` is `object` (mixed) because one value is the text `"1.25"`. Multiplying would fail.

In [5]:
six.dtypes

invoice         str
product         str
qty           int64
price        object
customer    float64
date            str
dtype: object

In [6]:
six["price"] = pd.to_numeric(six["price"], errors="coerce")
six["customer"] = six["customer"].astype(int)
six.dtypes

invoice         str
product         str
qty           int64
price       float64
customer      int64
date            str
dtype: object

## 5. Question 4: what is in the wrong format?

Dates are text in two different formats. The safe way is to parse each format *explicitly* and combine: `errors="coerce"` turns non-matching strings into `NaT`, and `fillna` takes the other attempt. (Pandas also offers `format="mixed"`, but guessing formats is exactly how `01/12/2010` silently becomes 12 January.)

In [7]:
iso = pd.to_datetime(six["date"], format="%Y-%m-%d", errors="coerce")
uk  = pd.to_datetime(six["date"], format="%d/%m/%Y", errors="coerce")
six["date"] = iso.fillna(uk)
six[["date"]].assign(weekday=six["date"].dt.day_name(), month=six["date"].dt.month)

,date,weekday,month
0,2010-12-01,Wednesday,12
2,2010-12-01,Wednesday,12
4,2010-12-01,Wednesday,12
5,2010-12-02,Thursday,12


## 6. The business rule: cancellations

Invoice starting with `C` means cancelled, and quantity is negative. Not a type problem, a *decision*. For a sales report we drop them.

In [8]:
is_cancelled = six["invoice"].str.startswith("C")
six = six[~is_cancelled]
six["revenue"] = six["qty"] * six["price"]
print("clean revenue: £", six["revenue"].sum().round(2))
six

clean revenue: £ 40.29


,invoice,product,qty,price,customer,date,revenue
0,536365,WHITE LANTERN,6,3.39,17850,2010-12-01,20.34
4,536381,JAM JAR,12,1.25,15311,2010-12-01,15.00
5,536382,TEA TOWEL,3,1.65,15311,2010-12-02,4.95


£40.29, matching the README. Now the same four questions on 541,910 rows.

## 7. The real data

In [9]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import sys, pandas as pd
sys.path.insert(0, str(data_path("download.py").parent))
from download import ensure_online_retail

raw = pd.read_excel(ensure_online_retail(), sheet_name="Year 2010-2011")
print(raw.shape)
raw.head()

(541910, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [10]:
raw.dtypes

Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
dtype: object

`Customer ID` is `float64`. Integers cannot hold `NaN`, so when some IDs are missing Pandas upgrades the whole column to float. That is why IDs show as `17850.0`. It is a *symptom* of missing values.

In [11]:
missing = pd.DataFrame({"count": raw.isna().sum(), "percent": raw.isna().mean().mul(100).round(1)})
missing[missing["count"] > 0]

,count,percent
Description,1454,0.3
Customer ID,135080,24.9


A quarter of rows have no customer: guest checkouts. Keep that number in your head; exercise 1 asks how much revenue they carry.

In [12]:
print("exact duplicate rows:", raw.duplicated().sum())
raw[raw.duplicated(keep=False)].sort_values(["Invoice", "StockCode"]).head(6)

exact duplicate rows: 5268


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom


Same invoice, product, quantity, timestamp: an export artefact, not two purchases. We keep the first of each.

In [13]:
print("cancellations (Invoice starts with C):", raw["Invoice"].astype(str).str.startswith("C").sum())
print("negative quantity:", (raw["Quantity"] < 0).sum())
print("price <= 0:", (raw["Price"] <= 0).sum())
raw[raw["Price"] <= 0].head(3)

cancellations (Invoice starts with C): 9288
negative quantity: 10624
price <= 0: 2517


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,NaN,United Kingdom
1510,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,NaN,United Kingdom
1985,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom


Zero-price rows are mostly missing-description rows: damaged stock, samples, adjustments. Not sales.

Dates are already `datetime64` because the file is Excel. From a CSV they would be text and you would call `pd.to_datetime`. Either way, the `.dt` accessor then unlocks the calendar.

In [14]:
print(raw["InvoiceDate"].min(), "→", raw["InvoiceDate"].max())
raw["InvoiceDate"].dt.day_name().value_counts()

2010-12-01 08:26:00 → 2011-12-09 12:50:00


InvoiceDate
Thursday     103857
Tuesday      101808
Monday        95111
Wednesday     94565
Friday        82194
Sunday        64375
Name: count, dtype: int64

No Saturdays. ShopWave's warehouse does not ship on Saturdays. You discovered that by looking at a dtype properly.

## 8. `clean_retail()`: the function you will reuse

All decisions in one place, written down. Every later lesson imports this logic.

In [15]:
def clean_retail(df):
    """The standard ShopWave cleaning used from lesson 01.5 onwards."""
    out = df.copy()
    out = out[~out["Invoice"].astype(str).str.startswith("C")]   # drop cancellations
    out = out[(out["Quantity"] > 0) & (out["Price"] > 0)]         # drop returns / free items
    out = out.dropna(subset=["Customer ID"])                       # drop guest checkouts
    out = out.drop_duplicates()
    out["Customer ID"] = out["Customer ID"].astype(int)
    out["Revenue"] = out["Quantity"] * out["Price"]
    return out

sales = clean_retail(raw)
print(f"{len(raw):,} rows → {len(sales):,} after cleaning")

541,910 rows → 392,693 after cleaning


In [16]:
sales.dtypes

Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID             int64
Country                   str
Revenue               float64
dtype: object

In [17]:
print(f"Revenue, cleaned: £{sales['Revenue'].sum():,.0f}")
print(f"Customers: {sales['Customer ID'].nunique():,}")
print(f"Products:  {sales['StockCode'].nunique():,}")

Revenue, cleaned: £8,887,227
Customers: 4,338
Products:  3,665


## Quiz answers

1. The *fraction* of missing values per column (multiply by 100 for a percentage).
2. Because some IDs are missing, and an integer column cannot hold `NaN`; Pandas promotes the column to float.
3. The "duplicates" may be real repeat purchases within one order; and you should know *why* duplicates exist (export bug? merge bug?) before deleting evidence.
4. It converts values that cannot be parsed into `NaN` instead of raising an error. Risk: it silently destroys data; count NaNs before and after.
5. Correct: `discount` missing → 0 (no discount). Wrong: `price` missing → 0 (makes the product look free and skews every average).

**Next:** [01.6 Groupby, merge, pivot](../06-pandas-groupby-merge/)